In [1]:
#Jets as graphs

In [5]:
import torch
import h5py
import numpy as np
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from sklearn.model_selection import train_test_split

In [3]:
file = h5py.File("quark-gluon_data-set_n139306.hdf5", "r")
print(list(file.keys()))

['X_jets', 'm0', 'pt', 'y']


In [6]:
images = file["X_jets"][:20000]   # load only 20k samples
labels = file["y"][:20000]

print(images.shape)

(20000, 125, 125, 3)


In [ ]:
images = np.transpose(images, (0,3,1,2))

print(images.shape)

In [ ]:
images = images / np.max(images)

In [ ]:
def image_to_pointcloud(image):

    nodes = []

    for c in range(3):
        for i in range(125):
            for j in range(125):
                val = image[c,i,j]
                if val > 0:
                    x = i / 125
                    y = j / 125
                    nodes.append([x, y, val, c])

    return np.array(nodes)

In [ ]:
from sklearn.neighbors import NearestNeighbors

def build_edges(nodes, k=5):

    coords = nodes[:, :2]

    nbrs = NearestNeighbors(n_neighbors=k).fit(coords)
    distances, indices = nbrs.kneighbors(coords)

    edges = []

    for i in range(len(nodes)):
        for j in indices[i]:
            edges.append([i, j])

    edge_index = torch.tensor(edges).t().contiguous()

    return edge_index

In [ ]:
from torch_geometric.data import Data

def create_graph(image, label):
    nodes = image_to_pointcloud(image)
    x = torch.tensor(nodes, dtype=torch.float)
    edge_index = build_edges(nodes)
    y = torch.tensor([int(label)], dtype=torch.long) 

    return Data(x=x, edge_index=edge_index, y=y)

In [ ]:
graphs = []

for i in range(1000):
    graph = create_graph(images[i], labels[i])
    graphs.append(graph)

In [ ]:
train_graphs, test_graphs = train_test_split(
    graphs, test_size=0.2, random_state=42
)

In [ ]:
from torch_geometric.loader import DataLoader

train_loader = DataLoader(train_graphs, batch_size=8, shuffle=True)
test_loader = DataLoader(test_graphs, batch_size=8)

In [ ]:
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool

class JetGNN(torch.nn.Module):

    def __init__(self):

        super().__init__()

        self.conv1 = GCNConv(4, 64)
        self.conv2 = GCNConv(64, 64)
        self.conv3 = GCNConv(64, 128)
        self.fc1 = torch.nn.Linear(128, 64)
        self.fc2 = torch.nn.Linear(64, 2)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = F.relu(self.conv3(x, edge_index))
        x = global_mean_pool(x, batch)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = JetGNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = torch.nn.CrossEntropyLoss()

In [ ]:
for epoch in range(10):

    model.train()
    total_loss = 0

    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data)
        loss = criterion(out, data.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print("Epoch:", epoch, "Loss:", total_loss)

In [ ]:
model.eval()

correct = 0
total = 0

for data in test_loader:
    data = data.to(device)
    out = model(data)
    pred = out.argmax(dim=1)
    correct += (pred == data.y).sum().item()
    total += data.y.size(0)

accuracy = correct / total
print("Test Accuracy:", accuracy)